This notebook runs the Watershed algorithm on the annotation file created by the Watershed-SV annotation pipeline.

This notebook should take about 3 minutes to run.


## Setup

In [1]:
library(AnVILGCP)
library(data.table)

# Watershed requirements
install.packages(c('PRROC','optparse','lbfgs'))

# Watershed algorithm code
anno_meta_tbl <- setDT(avtable('anno_metadata'))
watershed_folder <- anno_meta_tbl[file %in% grep('Watershed_with_tweaks/', file, val=T), file]
system(paste('gcloud storage cp -r', watershed_folder, '.'))

Installing packages into ‘/home/jupyter/packages’
(as ‘lib’ is unspecified)

also installing the dependency ‘getopt’


Registered S3 methods overwritten by 'AnVIL':
  method                         from    
  print.avworkflow_configuration AnVILGCP
  print.gcloud_sdk_result        AnVILGCP



In [2]:
subset_watershed_sv_output <- fread('subset_training_data.tsv')
  full_watershed_sv_output <- fread(  'full_training_data.tsv')
gene_sample_z       <- fread('mage_expression-watershed_sv_input.tsv')

# Define N2 Pairs
[See here](https://github.com/BennyStrobes/Watershed?tab=readme-ov-file#input-file) the format Watershed expects as input.

In [3]:
tmp <- subset_watershed_sv_output[, N2pair := as.integer(N2pair)
  ][, N2pair := fifelse(.N==2,.GRP,NA), by=GeneName
  ][, setcolorder(.SD, setdiff(names(.SD), 'N2pair'))
  ][, gene := NULL
]
tmp2 <- full_watershed_sv_output[, N2pair := as.integer(N2pair)
  ][, N2pair := fifelse(.N==2,.GRP,NA), by=GeneName
  ][, setcolorder(.SD, setdiff(names(.SD), 'N2pair'))
]

In [4]:
head(tmp)

SubjectID,GeneName,SVTYPE_DUP_gene_body,SVTYPE_DEL_gene_body,SVTYPE_INV_gene_body,SVTYPE_DUP_tss_flank,SVTYPE_DEL_tss_flank,SVTYPE_DUP_tes_flank,SVTYPE_DEL_tes_flank,af,⋯,is_ABC_SV_tes_flank,mean_GC_content_tes_flank,top10_LINSIGHT_tes_flank,top10_phastCON_tes_flank,top10_CADD_tes_flank,max_cpgpct_tes_flank,remap_crm_score_tes_flank,num_TADs_tes_flank,TE_pvalues_,N2pair
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
HG00146,ENSG00000174807:UW_VH_3482,0,0,0,0,0,0,1,0.00299521,⋯,0,48.35777,0.13788000,0.6911,13.5710,0.0,64,0,4.662432e-01,1
NA18553,ENSG00000168303:DUP_gs_CNV_7_40131223_40160568,1,0,0,1,0,0,0,0.00059904,⋯,0,0.00000,0.00000000,0.0000,0.0000,0.0,0,0,8.937764e-08,NA
HG03127,ENSG00000181192:DUP_gs_CNV_10_12111903_12124450,1,0,0,0,0,1,0,0.00638978,⋯,1,45.63265,0.51951611,0.7593,15.7500,0.0,418,5,9.404343e-02,NA
HG03127,ENSG00000181192:DUP_gs_CNV_10_12102547_12113247,1,0,0,0,0,0,0,0.00579073,⋯,0,0.00000,0.00000000,0.0000,0.0000,0.0,0,0,9.404343e-02,NA
HG03097,ENSG00000140470:DEL_pindel_43036,0,1,0,0,0,0,0,0.00658946,⋯,0,0.00000,0.00000000,0.0000,0.0000,0.0,0,0,5.012191e-01,NA
HG03118,ENSG00000162734:BI_GS_DEL1_B2_P0161_179,0,1,0,0,1,0,0,0.00798722,⋯,0,0.00000,0.00000000,0.0000,0.0000,0.0,0,0,1.841332e-01,NA
HG03084,ENSG00000138639:DEL_pindel_12414,0,1,0,0,0,0,0,0.00718850,⋯,0,0.00000,0.00000000,0.0000,0.0000,0.0,0,0,-5.629672e-01,NA
NA18522,ENSG00000082068:DEL_pindel_15235,0,0,0,0,1,0,0,0.00199681,⋯,0,0.00000,0.00000000,0.0000,0.0000,0.0,0,0,6.873047e-01,NA
NA19719,ENSG00000184226:DUP_uwash_chr13_66891604_66901672,0,1,0,0,0,0,0,0.00499201,⋯,0,0.00000,0.00000000,0.0000,0.0000,0.0,0,0,-1.902636e-01,NA


# Write

In [5]:
fwrite(tmp, 'subset_watershed_input.tsv', sep='\t', na=NA)
fwrite(tmp,   'full_watershed_input.tsv', sep='\t', na=NA)

In [6]:
system('gcloud storage cp "subset_watershed_input.tsv" "$WORKSPACE_BUCKET"')
system('gcloud storage cp   "full_watershed_input.tsv" "$WORKSPACE_BUCKET"')

# Run `predict_watershed.R`

In [7]:
old_wd <- setwd('Watershed_with_tweaks')

system(paste('Rscript predict_watershed.R',
  '--training_input    ../subset_watershed_input.tsv',
  '--prediction_input  ../subset_watershed_input.tsv',
  '--number_dimensions 1',
  '--output_prefix     watershed_output',
  '--model_name        Watershed_exact'
), intern=T)

system('gcloud storage cp watershed_output_* $WORKSPACE_BUCKET/data/derived/')

out <- fread('watershed_output_posterior_probability.txt')

setwd(old_wd)

[1] "Only RIVER can be run on data with 1 dimension."
 [2] " Changing model to RIVER."                      
 [3] "########################"                       
 [4] "ITERATION 1"                                    
 [5] "Average total norm: 0.063294138055469"          
 [6] "########################"                       
 [7] "ITERATION 2"                                    
 [8] "Average total norm: 0.0174892457352731"         
 [9] "########################"                       
[10] "ITERATION 3"                                    
[11] "Average total norm: 0.0143399376904429"         
[12] "########################"                       
[13] "ITERATION 4"                                    
[14] "Average total norm: 0.0130469152098088"         
[15] "########################"                       
[16] "ITERATION 5"                                    
[17] "Average total norm: 0.0121742538981155"         
[18] "########################"                       
[19] "ITERATION 6"                                    
[20] "Average total norm: 0.0135662690914277"         
[21] "########################"                       
[22] "ITERATION 7"                                    
[23] "Average total norm: 0.0226171970930195"         
[24] "########################"                       
[25] "ITERATION 8"                                    
[26] "Average total norm: 0.0137375660226994"         
[27] "########################"                       
[28] "ITERATION 9"                                    
[29] "Average total norm: 0.00733365484763117"        
[30] "########################"                       
[31] "ITERATION 10"                                   
[32] "Average total norm: 0.00542984047557571"        
[33] "########################"                       
[34] "ITERATION 11"                                   
[35] "Average total norm: 0.00436295605561169"        
[36] "########################"                       
[37] "ITERATION 12"                                   
[38] "Average total norm: 0.00351021761505342"        
[39] "########################"                       
[40] "ITERATION 13"                                   
[41] "Average total norm: 0.00282014734019624"        
[42] "########################"                       
[43] "ITERATION 14"                                   
[44] "Average total norm: 0.00225497089685684"        
[45] "########################"                       
[46] "ITERATION 15"                                   
[47] "Average total norm: 0.00181152538747159"        
[48] "########################"                       
[49] "ITERATION 16"                                   
[50] "Average total norm: 0.00147537622076324"        
[51] "########################"                       
[52] "ITERATION 17"                                   
[53] "Average total norm: 0.00116046463312169"        
[54] "########################"                       
[55] "ITERATION 18"                                   
[56] "Average total norm: 0.000920664898049844"       
[57] "########################"                       
[58] "ITERATION 19"                                   
[59] "Average total norm: 0.000756774747048177"       
[60] "########################"                       
[61] "ITERATION 20"                                   
[62] "Average total norm: 0.000616696119189067"       
[63] "########################"                       
[64] "ITERATION 21"                                   
[65] "Average total norm: 0.000499531485230761"       
[66] "########################"                       
[67] "ITERATION 22"                                   
[68] "Average total norm: 0.000406513332475786"       
[69] "########################"                       
[70] "ITERATION 23"                                   
[71] "Average total norm: 0.000332466600118931"       
[72] "########################"                       
[73] "ITERATION 24"                      

Below are the posterior probabilities output for each gene-individual pair:

In [8]:
out

sample_names,Watershed_posterior_outlier_signal_1,GAM_posterior_outlier_signal_1
<chr>,<dbl>,<dbl>
HG00146:ENSG00000174807:UW_VH_3482,3.335645e-03,0.1518288
NA18553:ENSG00000168303:DUP_gs_CNV_7_40131223_40160568,9.769242e-01,0.8751248
HG03127:ENSG00000181192:DUP_gs_CNV_10_12111903_12124450,9.907413e-01,0.9190249
HG03127:ENSG00000181192:DUP_gs_CNV_10_12102547_12113247,9.902834e-01,0.9659229
HG03097:ENSG00000140470:DEL_pindel_43036,4.878662e-03,0.1910408
HG03118:ENSG00000162734:BI_GS_DEL1_B2_P0161_179,3.171402e-02,0.3780300
HG03084:ENSG00000138639:DEL_pindel_12414,8.526750e-03,0.1194460
NA18522:ENSG00000082068:DEL_pindel_15235,5.245475e-03,0.0850104
NA19719:ENSG00000184226:DUP_uwash_chr13_66891604_66901672,3.506919e-02,0.2269041
